# Example 12.4.2 / 1: Mark a Decomposition for GCG

Companion notebook for **12.4.2 Dantzig-Wolfe Decomposition and GCG** in the AMPL lab Documentation, section "Mark a Decomposition for GCG". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Mark a Decomposition for GCG

GCG can search automatically for a decomposition. When the intended structure is known, the AMPL examples declare the input suffixes `master` and `block`. A constraint marked `suffix master 1` belongs to the master; a constraint marked `suffix block b` belongs to block $b$.

For example, in the bin-packing model, item-allocation rows are linking constraints and each bin-capacity row belongs to the block for that bin. The declarations are `suffix master IN, binary;` and `suffix block IN, integer;`. These annotations describe structure to GCG; they do not change the feasible set of the original AMPL model.

The model below is the bin-packing model of the AMPL Colab notebook, with the six-item instance.
The first run displays the partition with `ampl.display` and solves the original formulation with HiGHS,
once as a MIP and once with the AMPL option `relax_integrality` set to 1:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param n;                                     # number of items
param C;                                     # bin capacity

suffix master IN, binary;                    # 1 = row stays in the Dantzig-Wolfe master
suffix block IN, integer;                    # pricing block that owns the row

set I = 1..n ordered;                        # items to pack
param w {I} > 0;                             # item weight
param maxVal := max {i in I} w[i];           # heaviest item weight
param maxbins := ceil(n / floor(C / maxVal));  # upper bound on bins needed

set J = 1..maxbins;                          # candidate bins

var x {I,J} binary;                          # item i goes into bin j
var y {J} binary;                            # bin j is used

minimize Cost: sum {j in J} y[j];            # bins used

subject to b_Capacity {j in J}:              # one pricing block per bin
    sum {i in I} w[i] * x[i,j] <= C * y[j] suffix block j;

subject to m_Allocate {i in I}:              # linking rows: pack every item
    sum {j in J} x[i,j] >= 1 suffix master 1;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["n"] = 6
ampl.param["C"] = 10
ampl.param["w"] = {1: 6, 2: 5, 3: 4, 4: 4, 5: 3, 6: 2}

In [5]:
# run.py
ampl.display("m_Allocate.master", "b_Capacity.block")
ampl.solve(solver="highs", verbose=False)
print("HiGHS on the original rows:", ampl.solve_result, "bins", ampl.obj["Cost"].value())
ampl.option["relax_integrality"] = 1
ampl.solve(solver="highs", verbose=False)
print("compact LP relaxation:", ampl.obj["Cost"].value())

: m_Allocate.master b_Capacity.block    :=
1          1                1
2          1                2
3          1                3
4          1                4
5          1                5
6          1                6
;

HiGHS on the original rows: solved bins 3.0
compact LP relaxation: 2.4


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param n := 6;
param C := 10;
param w := 1 6  2 5  3 4  4 4  5 3  6 2;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display m_Allocate.master, b_Capacity.block;
option solver highs, solver_msg 0;
solve;
display solve_result, Cost;
option relax_integrality 1;
solve;
display Cost;

: m_Allocate.master b_Capacity.block    :=
1          1                1
2          1                2
3          1                3
4          1                4
5          1                5
6          1                6
;

solve_result = solved
Cost = 3

Cost = 2.4



### Another way in amplpy

The same model and data solved with GCG. `gcg_options` asks for the `.bestbound` and `.relmipgap`
suffixes, read with `ampl.get_value`; the second solve marks `b_Capacity[1]` as a master row with
`ampl.eval`, because setting a suffix value has no dedicated amplpy method:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.param["n"] = 6
ampl.param["C"] = 10
ampl.param["w"] = {1: 6, 2: 5, 3: 4, 4: 4, 5: 3, 6: 2}

In [11]:
# run.py
ampl.option["relax_integrality"] = 0
ampl.solve(solver="gcg", gcg_options="mip:bestbound=1 mip:return_gap=1", verbose=False)
print("GCG:", ampl.solve_result, "bins", ampl.obj["Cost"].value(),
      "bound", ampl.get_value("Cost.bestbound"), "gap", ampl.get_value("Cost.relmipgap"))

ampl.eval("let b_Capacity[1].master := 1; let b_Capacity[1].block := 0;")
ampl.solve(solver="gcg", verbose=False)
print("GCG with b_Capacity[1] in the master:", ampl.solve_result, "bins", ampl.obj["Cost"].value())

GCG: solved bins 3.0 bound 3 gap 0
GCG with b_Capacity[1] in the master: solved bins 3.0
